In [ ]:
# 실습 준비 — 12주차 통계적 추정 · 통계적 가설검정
# 이 셀을 먼저 한 번 실행하세요. 데이터가 없으면 아래 셀들이 전부 실패합니다.
import os, pathlib, urllib.request

BASE = "https://raw.githubusercontent.com/aprilslab/statistics-lab/main/data/"
FILES = ["ch4_scores400.csv", "ch10_enquete.csv", "ch10_access.csv"]

pathlib.Path("data").mkdir(exist_ok=True)
for name in FILES:
    for dest in (pathlib.Path(name), pathlib.Path("data") / name):
        if not dest.exists():
            urllib.request.urlretrieve(BASE + name, dest)

# '../data/x.csv' 로 읽는 노트북 대응 — 상위 폴더에도 같은 data/ 를 걸어둔다.
# 절대경로(/data)로 박으면 cwd 가 /content 가 아닐 때 깨지므로 상대경로로 건다.
try:
    parent = pathlib.Path("..") / "data"
    if not parent.exists():
        os.symlink(pathlib.Path("data").resolve(), parent)
except OSError:
    pass

print("준비 완료:", ", ".join(FILES) if FILES else "(내려받을 데이터 없음)")


# 12주차 · 통계적 추정 — 실습

수업 슬라이드의 코드를 슬라이드 순서대로 따라 합니다. 맨 위 준비 셀부터 차례로 실행하세요.

`{ }` 가 있는 셀은 빈칸을 채운 뒤 실행합니다. 셀 첫 주석에 무엇을 채울지 적어 두었고, 줄 끝 주석의 값이 슬라이드와 같은 결과입니다.

In [ ]:
# 준비 — 실행만 하세요
import numpy as np
import pandas as pd
from scipy import stats
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'   # 셀 안의 결과를 줄마다 모두 보인다
np.set_printoptions(legacy='1.25')                 # 70.4 처럼 숫자만 보인다

scores = np.array(pd.read_csv('ch4_scores400.csv')['score'])            # 학생 400명의 점수 = 모집단
enquete = np.array(pd.read_csv('ch10_enquete.csv')['known'])            # 설문 1000명 — 1 안다 · 0 모른다
n_access = np.array(pd.read_csv('ch10_access.csv')['access number'])    # 72시간의 시간당 접속 수

## 모수 · 추정량 · 추정값

모수란 모집단의 특징을 나타내는 값(모평균 μ, 모분산 σ²)이다. 추정량이란 모수를 추측하려고 표본으로 계산하는 식(예: 표본평균 X̄)이고, 실제 표본을 넣어 나온 값이 추정값이다. 교재는 학생 400명의 시험 점수 전체를 모집단으로 가정한다.

In [ ]:
# 슬라이드 3장 — 모수 · 추정량 · 추정값
p_mean = np.mean(scores)        # 69.53     모평균 μ   (scores = 400명 점수 = 모집단)
p_var  = np.var(scores)         # 206.669   모분산 σ²

np.random.seed(0)
sample = np.random.choice(scores, 20)   # A학생이 뽑은 표본 20개
np.mean(sample)                 # 70.4      표본평균 — μ 의 추정값

## 점추정 — 모집단을 알면 정답을 계산해 추정값과 비교한다

점추정이란 모평균이나 모분산 등의 모수를 하나의 수치로 추정하는 것이다(교재 266쪽). 실제 분석에서는 모집단 전체를 모르므로 μ · σ² 를 계산할 수 없고, 표본으로 추정한다. 교재는 연습을 위해 400명 전체를 모집단으로 두었다 — 그래서 μ · σ² 를 np.mean · np.var 로 직접 구해 정답으로 삼고, 표본 20개로 낸 추정값과 비교한다. 모분산의 추정값 s² 는 불편분산 — n 대신 n − 1 로 나눈 분산이다(이유는 다음 장).

In [ ]:
# 슬라이드 4장 — 점추정 — 모집단을 알면 정답을 계산해 추정값과 비교한다
p_mean = np.mean(scores)      # 69.53     모평균 μ — 400명 전체(모집단)를 알아서 직접 구한 정답
np.mean(sample)               # 70.4      x̄ — 표본 20개로 구한 μ 의 추정값
p_var = np.var(scores)        # 206.669   모분산 σ² — 모집단 전체라 그대로 n 으로 나눈다
np.var(sample, ddof=1)        # 158.25    s² — 표본 20개로 구한 σ² 의 추정값 (n − 1 로 나눔)

## 모분산의 추정값

A학생의 표본 sample(20개)로 모분산 σ² 의 추정값을 구합니다. 불편추정량이 되도록 빈칸을 채우세요.

In [ ]:
# 슬라이드 6장 — 모분산의 추정값
# 빈칸 { } — ddof 에 넣을 값 — n − 1 로 나누게
u_var = np.var(sample, { })

## 모평균의 구간추정 — 모분산을 알 때

언제: 모분산 σ² 를 안다(교재처럼 모집단 전체가 있을 때). 모집단이 정규분포면 정확하고, 아니면 n 이 충분히 클 때 중심극한정리로 쓰는 근사다. 구간은 x̄ ± 1.96 × σ/√n 이다. 1.96 은 표준정규분포 N(0, 1) 의 상위 2.5% 점이고, 양쪽에 2.5% 씩 남겨 가운데 95% 를 잡는다.

In [ ]:
# 준비 — 앞 장의 표본으로 구간추정에 쓸 값 (실행만 하세요)
n = len(sample)              # 20
s_mean = np.mean(sample)     # 70.4

In [ ]:
# 슬라이드 9장 — 모평균의 구간추정 — 모분산을 알 때
# 빈칸 { } — 표준정규분포의 상위 2.5% 점 — isf 에 넣을 위쪽 넓이
z = stats.norm().isf({ })        # 1.96    표준정규분포의 상위 2.5% 점
se = np.sqrt(p_var / n)            # 3.215   표준오차 σ/√n   (p_var = 모분산 σ²)
s_mean - z * se, s_mean + z * se   # (64.10, 76.70)   모평균의 95% 신뢰구간

## 모평균의 구간추정 — 모분산을 모를 때, t 분포

언제: 모분산을 모른다(현실의 대부분). σ 대신 표본으로 잰 s(불편분산 s² 의 제곱근)를 쓰고, 1.96 대신 자유도 n − 1 인 t 분포의 상위 2.5% 점을 쓴다. 모집단이 정규분포라는 가정이 필요하다. 자유도란 자유롭게 값을 취할 수 있는 변수의 개수다(교재 268쪽) — 불편분산은 편차의 합이 0 이라는 제약 때문에 n − 1 이다.

In [ ]:
# 슬라이드 14장 — 모평균의 구간추정 — 모분산을 모를 때, t 분포
# 빈칸 { } — t 분포의 자유도 — 표본 크기 n 에서 1 을 뺀다
t = stats.t(df={ }).isf(0.025)      # 2.093   자유도 19 인 t 분포의 상위 2.5% 점
se = np.sqrt(u_var / n)             # 2.813   표준오차 s/√n   (u_var = 불편분산 s²)
s_mean - t * se, s_mean + t * se    # (64.51, 76.29)

## 비율의 구간추정

언제: 응답이 1(예) · 0(아니오)인 데이터로 비율 p 를 추정한다. 표본비율 x̄ 는 1 의 개수를 n 으로 나눈 값, 곧 0 · 1 데이터의 평균이다. 표준오차는 √(x̄(1 − x̄)/n) 이고, n 이 클 때 쓰는 근사다.

In [ ]:
# 슬라이드 17장 — 비율의 구간추정
# 빈칸 { } — 비율의 표준오차 √(x̄(1 − x̄)/n)
n = len(enquete)                        # 1000
s_mean = enquete.mean()                 # 0.709    1000명 가운데 709명이 '안다'
se = { }  # 0.0144
s_mean - z * se, s_mean + z * se        # (0.681, 0.737)

z * np.sqrt(0.5 * 0.5 / 1000)           # 0.031    여론조사 1,000명의 '표본오차 ±3.1%p'

## 횟수의 평균 λ 의 구간추정 — 포아송 분포

언제: 한 시간의 접속 수처럼 횟수를 세는 데이터(포아송 분포 Poi(λ))로 평균 횟수 λ 를 추정한다. 포아송 분포는 평균과 분산이 모두 λ 라서 표준오차가 √(x̄/n) 이다. n 이 클 때 쓰는 근사다.

In [ ]:
# 슬라이드 18장 — 횟수의 평균 λ 의 구간추정 — 포아송 분포
# 빈칸 { } — 포아송의 표준오차 √(x̄/n)
n = len(n_access)                  # 72
s_mean = n_access.mean()          # 10.44   72시간, 한 시간 평균 접속 수
se = { }           # 0.381
s_mean - z * se, s_mean + z * se   # (9.70, 11.19)

## 모분산의 구간추정 — 카이제곱분포

언제: 모분산 σ² 의 범위를 추정한다. 모집단이 정규분포라는 가정이 필요하다. 이번에는 '추정값 ± 점 × 표준오차' 꼴이 아니라, (n − 1)s² 를 자유도 n − 1 인 카이제곱분포의 두 점으로 나눈다. 분포가 비대칭이라 구간도 s² 를 가운데 두지 않는다.

In [ ]:
# 슬라이드 19장 — 모분산의 구간추정 — 카이제곱분포
# 빈칸 { } — 카이제곱분포의 자유도 — n − 1
n = len(sample)                    # 20
rv = stats.chi2(df={ })
rv.isf(0.025), rv.isf(0.975)                       # (32.85, 8.91)   상위 · 하위 2.5% 점
(n-1) * u_var / rv.isf(0.025), (n-1) * u_var / rv.isf(0.975)   # (91.5, 337.6)